<a href="https://colab.research.google.com/github/leejunho12316/HonGongMachine/blob/main/ReACT_%EB%A9%80%ED%8B%B0%ED%94%8C_%EB%8F%84%EA%B5%AC_%EC%97%90%EC%9D%B4%EC%A0%84%ED%8A%B8.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#환경설정

In [5]:
!pip install -U "langchain>=1.0.0" "langchain-openai>=0.2.0" "langchain-community>=0.3.0" \
               "langchain-text-splitters>=0.2.0" "tiktoken>=0.7.0" "chromadb>=0.5.5" \
               "pymupdf>=1.24.0" "gradio>=4.44.0" "requests>=2.31.0"

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 5.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.2/87.2 kB 11.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 103.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.5/21.5 MB 61.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.9/24.9 MB 78.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.2/24.2 MB 21.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.3/58.3 kB 6.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.7/64.7 kB 5.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 31.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 95.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 61.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.1/17.1 MB 73.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB

In [1]:
import os
import re
import io
import sys
import json
import requests
from typing import Dict, List, Optional, Tuple
from contextlib import redirect_stdout

# LangChain 1.0 계열 임포트
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_community.document_loaders import PyMuPDFLoader
from langchain_community.vectorstores import Chroma
from langchain_core.tools import create_retriever_tool
from langchain_core.prompts import PromptTemplate

import gradio as gr

In [2]:
import os
os.environ['OPENAI_API_KEY'] = None

#데이터 로드

In [5]:
urls = [
   "https://raw.githubusercontent.com/llama-index-tutorial/llama-index-tutorial/main/ch06/ict_japan_2024.pdf",
   "https://raw.githubusercontent.com/llama-index-tutorial/llama-index-tutorial/main/ch06/ict_usa_2024.pdf"
]

# 각 파일 다운로드
for url in urls:
   filename = url.split("/")[-1]  # URL에서 파일명 추출
   response = requests.get(url)

   with open(filename, "wb") as f:
       f.write(response.content)
   print(f"{filename} 다운로드 완료")

ict_japan_2024.pdf 다운로드 완료
ict_usa_2024.pdf 다운로드 완료


#VectorDB 구현, 도구 생성

In [4]:
def create_pdf_retriever(
    pdf_path: str,  # PDF 파일 경로
    persist_directory: str,  # 벡터 스토어 저장 경로
    embedding_model: OpenAIEmbeddings,  # OpenAIEmbeddings 임베딩 모델
    chunk_size: int = 512,  # 청크 크기 기본값 512
    chunk_overlap: int = 0  # 청크 오버랩 크기 기본값 0
) -> Chroma.as_retriever:

    # PDF 파일로드
    loader = PyMuPDFLoader(pdf_path)
    data = loader.load()

    # 청킹
    text_splitter = RecursiveCharacterTextSplitter.from_tiktoken_encoder(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap
    )
    doc_splits = text_splitter.split_documents(data)

    # 벡터 스토어로 적재
    vectorstore = Chroma.from_documents(
        persist_directory=persist_directory,
        documents=doc_splits,
        embedding=embedding_model,
    )

    return vectorstore.as_retriever()

In [6]:
embd = OpenAIEmbeddings()

vectorDB 생성

In [9]:
retriever_japan = create_pdf_retriever(
    pdf_path = '/content/ict_japan_2024.pdf',
    persist_directory = 'db_ict_policy_japan_2024',
    embedding_model = embd
)

retriever_usa = create_pdf_retriever(
    pdf_path = '/content/ict_usa_2024.pdf',
    persist_directory = 'db_ict_policy_usa_2024',
    embedding_model = embd
)

도구 생성

In [10]:
jp_engine = create_retriever_tool(
    retriever=retriever_japan,
    name="japan_ict",
    description="일본의 ICT 시장동향 정보를 제공합니다. 일본 ICT와 관련된 질문은 해당 도구를 사용하세요.",
)

usa_engine = create_retriever_tool(
    retriever=retriever_usa,
    name="usa_ict",
    description="미국의 ICT 시장동향 정보를 제공합니다. 미국 ICT와 관련된 질문은 해당 도구를 사용하세요.",
)

In [11]:
tools = [jp_engine, usa_engine]
tool_map : Dict[str, object] = {t.name : t for t in tools}

In [14]:
tools

[StructuredTool(name='japan_ict', description='일본의 ICT 시장동향 정보를 제공합니다. 일본 ICT와 관련된 질문은 해당 도구를 사용하세요.', args_schema=<class 'langchain_core.tools.retriever.RetrieverInput'>, func=<function create_retriever_tool.<locals>.func at 0x78158e1ad440>, coroutine=<function create_retriever_tool.<locals>.afunc at 0x7815859cb560>),
 StructuredTool(name='usa_ict', description='미국의 ICT 시장동향 정보를 제공합니다. 미국 ICT와 관련된 질문은 해당 도구를 사용하세요.', args_schema=<class 'langchain_core.tools.retriever.RetrieverInput'>, func=<function create_retriever_tool.<locals>.func at 0x7815859cb1a0>, coroutine=<function create_retriever_tool.<locals>.afunc at 0x7815859cb240>)]

#ReACT 프롬프트

In [12]:
react_template = '''다음 질문에 최선을 다해 답변하세요. 당신은 다음 도구들에 접근할 수 있습니다:

{tools}

다음 형식을 사용하세요:

Question: 답변해야 하는 입력 질문
Thought: 무엇을 할지 항상 생각하세요
Action: 취해야 할 행동, [{tool_names}] 중 하나여야 합니다. 리스트에 있는 도구 중 1개를 택하십시오.
Action Input: 행동에 대한 입력값
Observation: 행동의 결과
... (이 Thought/Action/Action Input/Observation의 과정이 N번 반복될 수 있습니다)
Thought: 이제 최종 답변을 알겠습니다
Final Answer: 원래 입력된 질문에 대한 최종 답변 (한글로 작성하십시오.)

## 추가적인 주의사항
- 반드시 [Thought -> Action -> Action Input -> Observation] 순서를 준수하십시오. 항상 Action 전에는 Thought가 먼저 나와야 합니다.
- 최종 답변에는 최대한 많은 내용을 포함하십시오.
- 한 번의 검색으로 해결되지 않을 것 같다면 문제를 분할하여 푸는 것도 고려하십시오.
- 정보가 취합되었다면 불필요하게 사이클을 반복하지 마십시오.
- 묻지 않은 정보를 찾으려고 도구를 사용하지 마십시오.

시작하세요!

Question: {input}
{agent_scratchpad}'''

In [23]:
prompt = PromptTemplate.from_template(react_template)

STOP_SEQ = ["\nObservation:"]
llm = ChatOpenAI(model="gpt-4.1", temperature=0, stop = STOP_SEQ)

In [15]:
def _format_tools_for_prompt(ts: List[object]) -> Tuple[str, str]:
    lines, names = [], []
    for t in ts:
        names.append(t.name)
        desc = getattr(t, "description", "")
        lines.append(f"{t.name}: {desc}")
    return "\n".join(lines), ", ".join(names)

def _render_prompt(user_input: str, scratchpad: str) -> str:
    tools_str, tool_names = _format_tools_for_prompt(tools)
    return prompt.format(
        tools=tools_str,
        tool_names=tool_names,
        input=user_input,
        agent_scratchpad=scratchpad,
    )

In [17]:
# =========================
# ReAct 파서 및 실행 루프
# =========================
ACTION_RE = re.compile(r"^Action\s*:\s*(?P<tool>.+?)\s*$", re.MULTILINE)
ACTION_INPUT_RE = re.compile(r"^Action Input\s*:\s*(?P<input>.+?)\s*$", re.MULTILINE)
FINAL_ANSWER_RE = re.compile(r"Final Answer\s*:\s*(?P<final>[\s\S]+)$", re.IGNORECASE)

In [27]:
def _parse_action_and_input(text: str) -> Tuple[Optional[str], Optional[str]]:
    m_act = ACTION_RE.search(text)
    m_in = ACTION_INPUT_RE.search(text)
    if m_act and m_in:
      return m_act.group("tool").strip(), m_in.group("input").strip()

    m_final = FINAL_ANSWER_RE.search(text)
    if m_final:
        return "__FINAL__", m_final.group("final").strip()


    return None, None

In [19]:
def _observation_to_text(observation_obj) -> str:
    if isinstance(observation_obj, list):
        # Document 리스트일 수 있음
        def doc_to_str(d):
            try:
                meta = getattr(d, "metadata", {}) or {}
                src = meta.get("source") or meta.get("file_path") or ""
                txt = getattr(d, "page_content", "")
                if len(txt) > 500:
                    txt = txt[:500] + "..."
                return f"[source={src}] {txt}"
            except Exception:
                return str(d)
        return "\n".join(doc_to_str(d) for d in observation_obj[:5])
    return str(observation_obj)

In [25]:
def run_react(user_input: str, max_iters: int = 8) -> Dict[str, str]:
    scratchpad = ""  # 에이전트의 사고 과정을 누적 저장
    for _ in range(max_iters):
        # 프롬프트 조립 및 LLM 호출
        rendered = _render_prompt(user_input, scratchpad)
        resp = llm.invoke(rendered)
        text = resp.content if hasattr(resp, "content") else str(resp)

        # LLM 응답 파싱
        tool, action_input = _parse_action_and_input(text)

        # 파싱 실패: 형식 힌트 추가 후 재시도
        if tool is None:
            hint = "\n[파싱안내] 형식을 엄격히 따르세요. 반드시 'Action:'와 'Action Input:'을 한 줄씩 제공하십시오.\n"
            scratchpad += f"{text}\n{hint}"
            continue

        # 최종 답변 도달: 결과 반환 및 루프 종료
        if tool == "__FINAL__":
            final_answer = action_input
            return {"output": final_answer, "log": scratchpad + "\n" + text}

        # 존재하지 않는 도구: 에러 메시지 추가 후 재시도
        if tool not in tool_map:
            observation = f"[에러] 존재하지 않는 도구입니다: {tool}"
            scratchpad += f"{text}\nObservation: {observation}\n"
            continue

        # 도구 실행 및 결과 저장
        try:
            observation_obj = tool_map[tool].invoke(action_input)
            observation = _observation_to_text(observation_obj)
            scratchpad += f"{text}\nObservation: {observation}\n"
        except Exception as e:
            scratchpad += f"{text}\nObservation: [도구실행오류] {e}\n"

    # 최대 반복 횟수 초과
    return {
        "output": "반복 한도를 초과했습니다. 질문을 더 구체화해 주세요.",
        "log": scratchpad,
    }

In [28]:
q = "미국과 일본의 ICT 주요 정책의 공통점과 차이점을 설명해주고 미국의 블록체인 스타트업 지원 제도를 정리해 주세요."
out = run_react(q, max_iters=10)
print("최종 답변:", out["output"])
print("\n=== 실행 로그 (Thought / Action / Action Input / Observation) ===\n")
print(out["log"])

최종 답변: 미국과 일본의 ICT 주요 정책의 공통점과 차이점, 그리고 미국의 블록체인 스타트업 지원 제도는 다음과 같습니다.

### 1. 미국과 일본 ICT 주요 정책의 공통점

- **첨단 기술 육성**: 양국 모두 인공지능(AI), 반도체, 양자컴퓨팅, 6G 등 첨단 ICT 기술의 연구개발과 산업 육성에 정책적 역량을 집중하고 있습니다.
- **디지털 인프라 강화**: 디지털 사회 실현을 위한 인프라(데이터센터, 클라우드, 네트워크 등) 확충에 적극적입니다.
- **산업 경쟁력 강화**: 국가 차원에서 ICT 산업의 글로벌 경쟁력 확보를 위해 법령 및 규제 정비, 투자 촉진, 전략적 지원을 추진합니다.
- **국제 협력 확대**: 미국과 일본 모두 기술 교류 및 공동 연구 등 국가 간 협력을 강화하고 있습니다. 특히 양자컴퓨팅, 반도체 등에서 양국 간 협력이 활발합니다.
- **사이버보안 강화**: 사이버보안 위협에 대응하기 위한 정책과 기술 개발에 힘쓰고 있습니다.

### 2. 미국과 일본 ICT 주요 정책의 차이점

- **정책 추진 방식**: 
  - 미국은 민간 주도의 혁신과 정부의 대규모 재정 지원(예: CHIPS Act 등)을 결합하여 시장 중심의 정책을 펼칩니다.
  - 일본은 정부 주도의 전략 수립과 규제 완화, 산업경쟁력 강화법 등 법률 개정을 통한 체계적 지원이 특징입니다.
- **중점 분야**:
  - 미국은 AI, 반도체, 클라우드, 드론, 의료 AI, 우주 ICT 등 다양한 분야에서 선도적 역할을 강조합니다.
  - 일본은 디지털 행정, Web3(블록체인), 자체 소프트웨어 개발, 데이터센터 허브화 등 디지털 전환과 신산업 육성에 초점을 맞춥니다.
- **규제 환경**:
  - 미국은 혁신을 저해하지 않는 범위 내에서 규제 완화와 동시에, 국가 안보 및 개인정보 보호 등에서 엄격한 규제를 병행합니다.
  - 일본은 산업경쟁력 강화와 디지털 사회 실현을 위해 규제 완화와 지원 정책을 병행하지만, 전통적으로 보수적인 규제 환경이 남